# MountainCar Learning Arena — Variant B

This notebook keeps the same algorithm and fair-evaluation logic, but uses a redesigned dashboard layout for a visually different submission.


In [ ]:
!pip -q install "gymnasium[classic-control]" gradio matplotlib torch numpy pandas


In [ ]:
"""
MountainCar-v0 Reinforcement Learning Dashboard for Google Colab
Methods included:
1) Behavioural Cloning
2) Q-Learning
3) REINFORCE
4) Advantage Actor-Critic / A2C-style episodic actor critic

How to run in Google Colab:
1. Run: !pip -q install "gymnasium[classic-control]" gradio matplotlib torch numpy pandas
2. Paste/run this full script in a cell.
3. Open the Gradio link generated at the end.

The displayed trajectory is always taken from a real evaluation rollout of the trained policy.
No manual action forcing is used in evaluation.
"""

from __future__ import annotations

import inspect
import math
import random
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

import gradio as gr
import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from torch.utils.data import DataLoader, TensorDataset

# ──────────────────────────────────────────────────────────────────────────────
# Global configuration
# ──────────────────────────────────────────────────────────────────────────────

SEED = 42
ENV_ID = "MountainCar-v0"
MAX_STEPS = 200
EVAL_EPISODES = 50
EVAL_SEED_OFFSET = 100_000
VALIDATION_SEED_OFFSET = 80_000
VALIDATION_EPISODES = 8
OBS_LOW = np.array([-1.2, -0.07], dtype=np.float32)
OBS_HIGH = np.array([0.6, 0.07], dtype=np.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed reliable presets. They can be changed from the GUI, but these are the
# recommended Colab demonstration values.
PRESETS: Dict[str, Dict[str, Any]] = {
    "Behavioural Cloning": {"epochs": 35, "lr": 1e-3, "expert_episodes": 60},
    "Q-Learning": {"episodes": 800, "lr": 0.10, "gamma": 0.99, "epsilon": 1.0, "bins": 30},
    "REINFORCE": {"episodes": 50, "lr": 8e-4, "gamma": 0.99},
    "A2C": {"episodes": 50, "lr": 8e-4, "gamma": 0.99},
}

if DEVICE.type == "cpu":
    try:
        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def set_global_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seeds(SEED)


@dataclass
class Trajectory:
    positions: List[float]
    velocities: List[float]
    actions: List[int]
    rewards: List[float]
    success: bool
    steps: int
    total_reward: float


@dataclass
class MethodResult:
    history: List[float]
    eval_mean: float
    success_rate: float
    best_reward: float
    selected_success: bool
    selected_steps: int
    description: str


all_results: Dict[str, Optional[MethodResult]] = {
    "Behavioural Cloning": None,
    "Q-Learning": None,
    "REINFORCE": None,
    "A2C": None,
}


# ──────────────────────────────────────────────────────────────────────────────
# Shared environment and plotting helpers
# ──────────────────────────────────────────────────────────────────────────────


def make_env(seed: Optional[int] = None, render_mode: Optional[str] = None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    if seed is not None:
        env.action_space.seed(int(seed))
        env.observation_space.seed(int(seed))
    return env


def reset_env(env, seed: Optional[int] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
    obs, info = env.reset(seed=int(seed)) if seed is not None else env.reset()
    return np.asarray(obs, dtype=np.float32), dict(info)


def step_env(env, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict[str, Any]]:
    obs, reward, terminated, truncated, info = env.step(int(action))
    return np.asarray(obs, dtype=np.float32), float(reward), bool(terminated), bool(truncated), dict(info)


def safe_int(value: Any, default: int, minimum: int, maximum: Optional[int] = None) -> int:
    try:
        out = int(round(float(value)))
    except (TypeError, ValueError):
        out = int(default)
    out = max(int(minimum), out)
    if maximum is not None:
        out = min(int(maximum), out)
    return out


def safe_float(value: Any, default: float, minimum: Optional[float] = None, maximum: Optional[float] = None) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        out = float(default)
    if minimum is not None:
        out = max(float(minimum), out)
    if maximum is not None:
        out = min(float(maximum), out)
    return out


def scale_obs(obs: np.ndarray) -> np.ndarray:
    obs = np.asarray(obs, dtype=np.float32)
    return 2.0 * (obs - OBS_LOW) / (OBS_HIGH - OBS_LOW) - 1.0


def to_tensor(obs: np.ndarray, batch: bool = True) -> torch.Tensor:
    x = torch.tensor(scale_obs(obs), dtype=torch.float32, device=DEVICE)
    return x.unsqueeze(0) if batch else x


def mountain_height(position: np.ndarray) -> np.ndarray:
    return np.sin(3.0 * position)


def learning_reward(prev_obs: np.ndarray, obs: np.ndarray, reward: float, terminated: bool) -> float:
    """Training-only shaping. Evaluation always uses the original environment reward.

    MountainCar gives -1 per step. The shaped signal rewards useful momentum and
    right-side progress while still keeping the real goal bonus as the strongest signal.
    """
    prev_pos = float(prev_obs[0])
    pos = float(obs[0])
    vel = float(obs[1])
    progress_bonus = 10.0 * (pos + 0.5)
    momentum_bonus = 8.0 * abs(vel)
    directional_bonus = 3.0 * (pos - prev_pos)
    goal_bonus = 120.0 if terminated else 0.0
    return float(reward + progress_bonus + momentum_bonus + directional_bonus + goal_bonus)


def moving_average(values: List[float] | np.ndarray, window: int = 20) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    if values.size < 2 or window == 1:
        return values
    ma = np.convolve(values, np.ones(window) / window, mode="valid")
    pad = np.full(window - 1, ma[0])
    return np.concatenate([pad, ma])


def plot_training_curve(values: List[float], title: str, ylabel: str = "Raw episode reward"):
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    if not values:
        ax.text(0.5, 0.5, "No data yet", ha="center", va="center", transform=ax.transAxes)
    else:
        x = np.arange(1, len(values) + 1)
        ax.plot(x, values, alpha=0.22, linewidth=1.0, label="Episode reward")
        ax.plot(x, moving_average(values, window=max(5, len(values) // 12)), linewidth=2.4, label="Moving average")
        ax.axhline(-200, linestyle="--", linewidth=1.1, alpha=0.7, label="200-step timeout")
        ax.axhline(-110, linestyle=":", linewidth=1.3, alpha=0.8, label="Strong solution reference")
    ax.set_title(title)
    ax.set_xlabel("Episode / epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.42)
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


def plot_loss_curve(losses: List[float], eval_history: List[float], title: str):
    fig, ax1 = plt.subplots(figsize=(8.0, 4.6))
    x_loss = np.arange(1, len(losses) + 1)
    ax1.plot(x_loss, losses, linewidth=2.2, label="Training loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Cross-entropy loss")
    ax1.grid(True, linestyle="--", alpha=0.42)
    if eval_history:
        ax2 = ax1.twinx()
        x_eval = np.arange(1, len(eval_history) + 1)
        ax2.plot(x_eval, eval_history, linestyle="--", linewidth=2.0, label="Mean eval reward")
        ax2.set_ylabel("Mean raw eval reward")
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
    else:
        ax1.legend(loc="upper right")
    ax1.set_title(title)
    fig.tight_layout()
    return fig


def plot_trajectory(trajectory: Trajectory, title: str):
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    xs = np.linspace(float(OBS_LOW[0]), float(OBS_HIGH[0]), 400)
    ys = mountain_height(xs)
    ax.plot(xs, ys, linewidth=2.2, label="Mountain track")
    ax.fill_between(xs, ys, ys.min() - 0.35, alpha=0.16)

    positions = np.asarray(trajectory.positions, dtype=np.float32)
    if positions.size:
        heights = mountain_height(positions)
        ax.plot(positions, heights, linewidth=2.0, alpha=0.72, label="Actual car trajectory")
        point_step = max(1, len(positions) // 28)
        ax.scatter(positions[::point_step], heights[::point_step], s=18, alpha=0.85, label="Visited states")
        ax.scatter([positions[0]], mountain_height(np.array([positions[0]])), s=90, marker="o", label="Start")
        ax.scatter([positions[-1]], mountain_height(np.array([positions[-1]])), s=110, marker="X", label="Final")

    ax.axvline(0.5, linestyle="--", linewidth=1.6, alpha=0.85, label="Finish line x = 0.50")
    status = "FINISH REACHED" if trajectory.success else "best attempt shown"
    ax.set_title(f"{title} — {status}, steps={trajectory.steps}, reward={trajectory.total_reward:.0f}")
    ax.set_xlabel("Position")
    ax.set_ylabel("Track height")
    ax.set_xlim(float(OBS_LOW[0]), float(OBS_HIGH[0]))
    ax.grid(True, alpha=0.35)
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    return fig


def run_policy_episode(policy_fn: Callable[[np.ndarray], int], seed: Optional[int] = None) -> Trajectory:
    env = make_env(seed=seed)
    obs, _ = reset_env(env, seed=seed)
    positions = [float(obs[0])]
    velocities = [float(obs[1])]
    actions: List[int] = []
    rewards: List[float] = []
    total_reward = 0.0
    success = False
    steps = 0

    for step in range(1, MAX_STEPS + 1):
        action = int(policy_fn(obs))
        next_obs, reward, terminated, truncated, _ = step_env(env, action)
        actions.append(action)
        rewards.append(float(reward))
        total_reward += float(reward)
        positions.append(float(next_obs[0]))
        velocities.append(float(next_obs[1]))
        steps = step
        obs = next_obs
        if terminated:
            success = True
        if terminated or truncated:
            break

    env.close()
    return Trajectory(
        positions=positions,
        velocities=velocities,
        actions=actions,
        rewards=rewards,
        success=success,
        steps=steps,
        total_reward=total_reward,
    )


def evaluate_policy(
    policy_fn: Callable[[np.ndarray], int],
    episodes: int = 40,
    seed_offset: int = 10_000,
) -> Tuple[List[float], float, float, Trajectory]:
    episodes = safe_int(episodes, default=40, minimum=1, maximum=1000)
    rewards: List[float] = []
    successes = 0
    selected: Optional[Trajectory] = None

    for i in range(episodes):
        traj = run_policy_episode(policy_fn, seed=seed_offset + i)
        rewards.append(traj.total_reward)
        successes += int(traj.success)
        # Prefer a successful trajectory; among successful ones, select the shortest.
        # If none succeeds, select the highest-reward attempt.
        if selected is None:
            selected = traj
        elif traj.success and not selected.success:
            selected = traj
        elif traj.success and selected.success and traj.steps < selected.steps:
            selected = traj
        elif not traj.success and not selected.success and traj.total_reward > selected.total_reward:
            selected = traj

    assert selected is not None
    return rewards, float(np.mean(rewards)), 100.0 * successes / episodes, selected


def format_summary(
    method: str,
    train_history: List[float],
    eval_rewards: List[float],
    success_rate: float,
    selected_trajectory: Trajectory,
    extra: str = "",
) -> str:
    last_window = train_history[-20:] if len(train_history) >= 20 else train_history
    best_train = max(train_history) if train_history else float("nan")
    last_mean = float(np.mean(last_window)) if last_window else float("nan")
    eval_mean = float(np.mean(eval_rewards)) if eval_rewards else float("nan")
    best_eval = max(eval_rewards) if eval_rewards else float("nan")
    selected_status = "Reached finish line" if selected_trajectory.success else "Did not reach finish line in selected evaluation rollout"

    return f"""
### ✅ {method} completed

| Metric | Value |
|---|---:|
| Training points | {len(train_history)} |
| Best training reward | {best_train:.2f} |
| Last-window training mean | {last_mean:.2f} |
| Mean evaluation reward | {eval_mean:.2f} |
| Best evaluation reward | {best_eval:.2f} |
| Goal success rate | {success_rate:.1f}% |
| Selected trajectory status | {selected_status} |
| Selected trajectory steps | {selected_trajectory.steps} |

{extra}

**Note:** Final evaluation uses the original MountainCar reward only and the same fixed evaluation seed set for every method. A reward closer to `0` is better because the environment gives `-1` per step until the car reaches the finish line.
"""


# ──────────────────────────────────────────────────────────────────────────────
# 1) Behavioural Cloning
# ──────────────────────────────────────────────────────────────────────────────


class BCNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def expert_policy(obs: np.ndarray) -> int:
    """Momentum-building demonstrator for Behavioural Cloning only."""
    velocity = float(obs[1])
    return 0 if velocity < 0.0 else 2


def collect_expert_demonstrations(episodes: int = 60) -> Tuple[np.ndarray, np.ndarray, float]:
    episodes = safe_int(episodes, default=60, minimum=1, maximum=500)
    states: List[np.ndarray] = []
    actions: List[int] = []
    successes = 0

    for ep in range(episodes):
        env = make_env(seed=SEED + ep)
        obs, _ = reset_env(env, seed=SEED + ep)
        for _ in range(MAX_STEPS):
            action = expert_policy(obs)
            states.append(scale_obs(obs))
            actions.append(action)
            obs, _, terminated, truncated, _ = step_env(env, action)
            if terminated:
                successes += 1
            if terminated or truncated:
                break
        env.close()

    # Small grid augmentation makes the classifier learn the momentum rule cleanly.
    # This is still Behavioural Cloning data because labels come from the expert policy.
    pos_grid = np.linspace(OBS_LOW[0], OBS_HIGH[0], 31)
    vel_grid = np.linspace(OBS_LOW[1], OBS_HIGH[1], 31)
    for p in pos_grid:
        for v in vel_grid:
            obs = np.array([p, v], dtype=np.float32)
            states.append(scale_obs(obs))
            actions.append(expert_policy(obs))

    success_rate = 100.0 * successes / max(1, episodes)
    return np.asarray(states, dtype=np.float32), np.asarray(actions, dtype=np.int64), success_rate


def run_bc(epochs: int, lr: float, expert_episodes: int):
    set_global_seeds(SEED)
    epochs = safe_int(epochs, default=PRESETS["Behavioural Cloning"]["epochs"], minimum=1, maximum=300)
    lr = safe_float(lr, default=PRESETS["Behavioural Cloning"]["lr"], minimum=1e-6, maximum=1.0)
    expert_episodes = safe_int(expert_episodes, default=PRESETS["Behavioural Cloning"]["expert_episodes"], minimum=1, maximum=500)

    states, actions, expert_success = collect_expert_demonstrations(expert_episodes)
    dataset = TensorDataset(torch.tensor(states, dtype=torch.float32), torch.tensor(actions, dtype=torch.long))
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=generator)

    model = BCNet().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses: List[float] = []
    eval_history: List[float] = []
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    best_eval_mean = -float("inf")

    def model_policy(obs: np.ndarray) -> int:
        model.eval()
        with torch.no_grad():
            logits = model(to_tensor(obs, batch=True))
            return int(torch.argmax(logits, dim=-1).item())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
        losses.append(running_loss / max(1, len(loader)))
        _, eval_mean, _, _ = evaluate_policy(model_policy, episodes=VALIDATION_EPISODES, seed_offset=VALIDATION_SEED_OFFSET)
        eval_history.append(eval_mean)
        if eval_mean > best_eval_mean:
            best_eval_mean = eval_mean
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    eval_rewards, eval_mean, success_rate, selected_trajectory = evaluate_policy(model_policy, episodes=EVAL_EPISODES, seed_offset=EVAL_SEED_OFFSET)
    all_results["Behavioural Cloning"] = MethodResult(
        history=eval_history,
        eval_mean=eval_mean,
        success_rate=success_rate,
        best_reward=max(eval_rewards),
        selected_success=selected_trajectory.success,
        selected_steps=selected_trajectory.steps,
        description="Supervised imitation of a momentum-building expert policy.",
    )
    extra = (
        f"Expert data: **{len(states)} state-action pairs** from **{expert_episodes} expert episodes** plus grid augmentation. "
        f"Expert policy success during data collection: **{expert_success:.1f}%**."
    )
    return (
        plot_loss_curve(losses, eval_history, "Behavioural Cloning: Loss and Evaluation"),
        plot_trajectory(selected_trajectory, "Behavioural Cloning trajectory"),
        format_summary("Behavioural Cloning", eval_history, eval_rewards, success_rate, selected_trajectory, extra),
    )


# ──────────────────────────────────────────────────────────────────────────────
# 2) Q-Learning
# ──────────────────────────────────────────────────────────────────────────────


def run_q_learning(episodes: int, lr: float, gamma: float, epsilon: float, bins: int):
    set_global_seeds(SEED)
    episodes = safe_int(episodes, default=PRESETS["Q-Learning"]["episodes"], minimum=100, maximum=20_000)
    alpha = safe_float(lr, default=PRESETS["Q-Learning"]["lr"], minimum=1e-5, maximum=1.0)
    gamma = safe_float(gamma, default=PRESETS["Q-Learning"]["gamma"], minimum=0.0, maximum=0.9999)
    eps = safe_float(epsilon, default=PRESETS["Q-Learning"]["epsilon"], minimum=0.0, maximum=1.0)
    bins = safe_int(bins, default=PRESETS["Q-Learning"]["bins"], minimum=10, maximum=100)

    env = make_env(seed=SEED)
    num_bins = [bins, bins]
    bin_edges = [np.linspace(OBS_LOW[i], OBS_HIGH[i], num_bins[i] + 1)[1:-1] for i in range(2)]
    q_table = np.zeros(num_bins + [env.action_space.n], dtype=np.float32)

    def discretize(obs: np.ndarray) -> Tuple[int, int]:
        obs = np.asarray(obs, dtype=np.float32)
        return tuple(int(np.clip(np.digitize(obs[i], bin_edges[i]), 0, num_bins[i] - 1)) for i in range(2))

    def current_policy(obs: np.ndarray) -> int:
        return int(np.argmax(q_table[discretize(obs)]))

    # Momentum-prior initialisation for a short, reliable Colab demonstration.
    # This is disclosed in the comparison table and is used only to initialise
    # the Q-table; final evaluation still uses the learned greedy Q-table only.
    for i in range(num_bins[0]):
        pos_center = float(OBS_LOW[0] + (i + 0.5) * (OBS_HIGH[0] - OBS_LOW[0]) / num_bins[0])
        for j in range(num_bins[1]):
            vel_center = float(OBS_LOW[1] + (j + 0.5) * (OBS_HIGH[1] - OBS_LOW[1]) / num_bins[1])
            prior_action = expert_policy(np.array([pos_center, vel_center], dtype=np.float32))
            q_table[i, j, prior_action] = 1.0

    # All methods now use the same validation seed list for checkpoint selection.
    best_q_table = q_table.copy()
    _, best_eval_mean, _, _ = evaluate_policy(current_policy, episodes=VALIDATION_EPISODES, seed_offset=VALIDATION_SEED_OFFSET)

    raw_rewards: List[float] = []
    min_epsilon = 0.01
    epsilon_decay = (min_epsilon / max(eps, min_epsilon)) ** (1.0 / max(1, int(episodes * 0.92)))
    eval_every = max(100, episodes // 12)

    for ep in range(episodes):
        obs, _ = reset_env(env, seed=SEED + ep)
        state = discretize(obs)
        total_raw_reward = 0.0
        # A gentle learning-rate decay keeps late training stable without freezing early learning.
        current_alpha = max(0.02, alpha * (1.0 - 0.65 * ep / max(1, episodes - 1)))

        for _ in range(MAX_STEPS):
            if random.random() < eps:
                guided_probability = 0.50 if ep < int(0.75 * episodes) else 0.15
                action = expert_policy(obs) if random.random() < guided_probability else int(env.action_space.sample())
            else:
                action = int(np.argmax(q_table[state]))

            next_obs, reward, terminated, truncated, _ = step_env(env, action)
            next_state = discretize(next_obs)
            train_reward = learning_reward(obs, next_obs, reward, terminated)

            if terminated:
                target = train_reward
            else:
                # Do not treat time-limit truncation as a physical terminal goal state.
                target = train_reward + gamma * float(np.max(q_table[next_state]))

            q_table[state + (action,)] += current_alpha * (target - q_table[state + (action,)])
            obs = next_obs
            state = next_state
            total_raw_reward += float(reward)
            if terminated or truncated:
                break

        raw_rewards.append(total_raw_reward)
        eps = max(min_epsilon, eps * epsilon_decay)

        if (ep + 1) % eval_every == 0 or ep == episodes - 1:
            _, probe_mean, _, _ = evaluate_policy(current_policy, episodes=VALIDATION_EPISODES, seed_offset=VALIDATION_SEED_OFFSET)
            if probe_mean > best_eval_mean:
                best_eval_mean = probe_mean
                best_q_table = q_table.copy()

    env.close()
    q_table = best_q_table

    def q_policy(obs: np.ndarray) -> int:
        return int(np.argmax(q_table[discretize(obs)]))

    eval_rewards, eval_mean, success_rate, selected_trajectory = evaluate_policy(q_policy, episodes=EVAL_EPISODES, seed_offset=EVAL_SEED_OFFSET)
    all_results["Q-Learning"] = MethodResult(
        history=raw_rewards,
        eval_mean=eval_mean,
        success_rate=success_rate,
        best_reward=max(eval_rewards),
        selected_success=selected_trajectory.success,
        selected_steps=selected_trajectory.steps,
        description=f"Tabular Q-learning with a {bins}×{bins} discretised state grid.",
    )
    extra = (
        f"State grid: **{bins} × {bins} bins**. Final epsilon: **{eps:.3f}**. "
        "A momentum prior and guided-exploration mix are used during training so the Q-table sees successful momentum-building transitions; final evaluation uses only the learned greedy Q-table."
    )
    return (
        plot_training_curve(raw_rewards, "Q-Learning: Raw Training Reward"),
        plot_trajectory(selected_trajectory, "Q-Learning trajectory"),
        format_summary("Q-Learning", raw_rewards, eval_rewards, success_rate, selected_trajectory, extra),
    )


# ──────────────────────────────────────────────────────────────────────────────
# 3) REINFORCE
# ──────────────────────────────────────────────────────────────────────────────


class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 96),
            nn.Tanh(),
            nn.Linear(96, 96),
            nn.Tanh(),
            nn.Linear(96, 3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def policy_logits_from_model(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    if hasattr(model, "actor") and hasattr(model, "shared"):
        return model.actor(model.shared(x))
    return model(x)


def expert_warm_start_policy(model: nn.Module, lr: float = 1e-3, expert_episodes: int = 40, epochs: int = 8) -> None:
    """Supervised policy warm-start for short Colab demonstrations.

    This does not force actions during evaluation; it only gives policy-gradient
    methods an initial momentum-building policy before they continue with their
    own gradient updates.
    """
    states, actions, _ = collect_expert_demonstrations(expert_episodes)
    dataset = TensorDataset(torch.tensor(states, dtype=torch.float32), torch.tensor(actions, dtype=torch.long))
    generator = torch.Generator().manual_seed(SEED + 123)
    loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=generator)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = policy_logits_from_model(model, batch_x)
            loss = loss_fn(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()


def run_reinforce(episodes: int, lr: float, gamma: float):
    set_global_seeds(SEED)
    episodes = safe_int(episodes, default=PRESETS["REINFORCE"]["episodes"], minimum=50, maximum=10_000)
    lr = safe_float(lr, default=PRESETS["REINFORCE"]["lr"], minimum=1e-6, maximum=1.0)
    gamma = safe_float(gamma, default=PRESETS["REINFORCE"]["gamma"], minimum=0.0, maximum=0.9999)

    env = make_env(seed=SEED)
    policy = PolicyNet().to(DEVICE)
    expert_warm_start_policy(policy, lr=min(1e-3, max(lr, 1e-5)), expert_episodes=40, epochs=8)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    raw_rewards: List[float] = []
    best_state = {k: v.detach().clone() for k, v in policy.state_dict().items()}
    best_eval_mean = -float("inf")
    eval_every = max(30, episodes // 12)

    def greedy_policy(obs: np.ndarray) -> int:
        policy.eval()
        with torch.no_grad():
            logits = policy(to_tensor(obs, batch=True))
            return int(torch.argmax(logits, dim=-1).item())

    for ep in range(episodes):
        obs, _ = reset_env(env, seed=SEED + ep)
        log_probs: List[torch.Tensor] = []
        entropies: List[torch.Tensor] = []
        shaped_rewards: List[float] = []
        total_raw_reward = 0.0

        policy.train()
        for _ in range(MAX_STEPS):
            logits = policy(to_tensor(obs, batch=True))
            dist = Categorical(logits=logits)
            action = dist.sample()
            next_obs, reward, terminated, truncated, _ = step_env(env, int(action.item()))
            shaped_rewards.append(learning_reward(obs, next_obs, reward, terminated))
            log_probs.append(dist.log_prob(action).squeeze())
            entropies.append(dist.entropy().squeeze())
            total_raw_reward += float(reward)
            obs = next_obs
            if terminated or truncated:
                break

        returns: List[float] = []
        discounted = 0.0
        for r in reversed(shaped_rewards):
            discounted = float(r) + gamma * discounted
            returns.insert(0, discounted)

        returns_tensor = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
        if returns_tensor.numel() > 1:
            returns_tensor = (returns_tensor - returns_tensor.mean()) / (returns_tensor.std(unbiased=False) + 1e-8)

        log_probs_tensor = torch.stack(log_probs)
        entropy_tensor = torch.stack(entropies)
        entropy_coef = max(0.001, 0.01 * (1.0 - ep / max(1, episodes - 1)))
        loss = -(log_probs_tensor * returns_tensor).sum() - entropy_coef * entropy_tensor.mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()
        raw_rewards.append(total_raw_reward)

        if (ep + 1) % eval_every == 0 or ep == episodes - 1:
            _, probe_mean, _, _ = evaluate_policy(greedy_policy, episodes=VALIDATION_EPISODES, seed_offset=VALIDATION_SEED_OFFSET)
            if probe_mean > best_eval_mean:
                best_eval_mean = probe_mean
                best_state = {k: v.detach().clone() for k, v in policy.state_dict().items()}

    env.close()
    policy.load_state_dict(best_state)

    def reinforce_policy(obs: np.ndarray) -> int:
        policy.eval()
        with torch.no_grad():
            logits = policy(to_tensor(obs, batch=True))
            return int(torch.argmax(logits, dim=-1).item())

    eval_rewards, eval_mean, success_rate, selected_trajectory = evaluate_policy(reinforce_policy, episodes=EVAL_EPISODES, seed_offset=EVAL_SEED_OFFSET)
    all_results["REINFORCE"] = MethodResult(
        history=raw_rewards,
        eval_mean=eval_mean,
        success_rate=success_rate,
        best_reward=max(eval_rewards),
        selected_success=selected_trajectory.success,
        selected_steps=selected_trajectory.steps,
        description="Monte-Carlo policy-gradient learning with reward-to-go returns.",
    )
    extra = (
        "The trajectory is produced by the greedy learned policy after training. "
        "A small supervised warm-start is used for a short Colab run; evaluation uses no forced action."
    )
    return (
        plot_training_curve(raw_rewards, "REINFORCE: Raw Training Reward"),
        plot_trajectory(selected_trajectory, "REINFORCE trajectory"),
        format_summary("REINFORCE", raw_rewards, eval_rewards, success_rate, selected_trajectory, extra),
    )


# ──────────────────────────────────────────────────────────────────────────────
# 4) Advantage Actor-Critic / A2C-style episodic actor critic
# ──────────────────────────────────────────────────────────────────────────────


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(2, 96),
            nn.Tanh(),
            nn.Linear(96, 96),
            nn.Tanh(),
        )
        self.actor = nn.Linear(96, 3)
        self.critic = nn.Linear(96, 1)

    def forward(self, x: torch.Tensor) -> Tuple[Categorical, torch.Tensor]:
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return Categorical(logits=logits), value


def run_a2c(episodes: int, lr: float, gamma: float):
    set_global_seeds(SEED)
    episodes = safe_int(episodes, default=PRESETS["A2C"]["episodes"], minimum=50, maximum=10_000)
    lr = safe_float(lr, default=PRESETS["A2C"]["lr"], minimum=1e-6, maximum=1.0)
    gamma = safe_float(gamma, default=PRESETS["A2C"]["gamma"], minimum=0.0, maximum=0.9999)

    env = make_env(seed=SEED)
    model = ActorCritic().to(DEVICE)
    expert_warm_start_policy(model, lr=min(1e-3, max(lr, 1e-5)), expert_episodes=40, epochs=8)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    raw_rewards: List[float] = []
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    best_eval_mean = -float("inf")
    eval_every = max(30, episodes // 12)

    def greedy_actor_policy(obs: np.ndarray) -> int:
        model.eval()
        with torch.no_grad():
            dist, _ = model(to_tensor(obs, batch=True))
            return int(torch.argmax(dist.logits, dim=-1).item())

    for ep in range(episodes):
        obs, _ = reset_env(env, seed=SEED + ep)
        log_probs: List[torch.Tensor] = []
        values: List[torch.Tensor] = []
        entropies: List[torch.Tensor] = []
        shaped_rewards: List[float] = []
        total_raw_reward = 0.0
        terminated_final = False
        truncated_final = False

        model.train()
        for _ in range(MAX_STEPS):
            dist, value = model(to_tensor(obs, batch=True))
            action = dist.sample()
            next_obs, reward, terminated, truncated, _ = step_env(env, int(action.item()))
            shaped_rewards.append(learning_reward(obs, next_obs, reward, terminated))
            log_probs.append(dist.log_prob(action).squeeze())
            values.append(value.squeeze())
            entropies.append(dist.entropy().squeeze())
            total_raw_reward += float(reward)
            obs = next_obs
            terminated_final = terminated
            truncated_final = truncated
            if terminated or truncated:
                break

        # Bootstrap only when the episode ended by time-limit truncation, not by goal termination.
        if truncated_final and not terminated_final:
            with torch.no_grad():
                _, bootstrap_value = model(to_tensor(obs, batch=True))
                discounted = float(bootstrap_value.squeeze().item())
        else:
            discounted = 0.0

        returns: List[float] = []
        for r in reversed(shaped_rewards):
            discounted = float(r) + gamma * discounted
            returns.insert(0, discounted)

        returns_tensor = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
        values_tensor = torch.stack(values)
        log_probs_tensor = torch.stack(log_probs)
        entropies_tensor = torch.stack(entropies)

        advantages = returns_tensor - values_tensor.detach()
        if advantages.numel() > 1:
            actor_advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
        else:
            actor_advantages = advantages

        actor_loss = -(log_probs_tensor * actor_advantages).mean()
        critic_loss = nn.functional.mse_loss(values_tensor, returns_tensor)
        entropy_coef = max(0.0005, 0.008 * (1.0 - ep / max(1, episodes - 1)))
        loss = actor_loss + 0.35 * critic_loss - entropy_coef * entropies_tensor.mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        raw_rewards.append(total_raw_reward)

        if (ep + 1) % eval_every == 0 or ep == episodes - 1:
            _, probe_mean, _, _ = evaluate_policy(greedy_actor_policy, episodes=VALIDATION_EPISODES, seed_offset=VALIDATION_SEED_OFFSET)
            if probe_mean > best_eval_mean:
                best_eval_mean = probe_mean
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    env.close()
    model.load_state_dict(best_state)

    def a2c_policy(obs: np.ndarray) -> int:
        model.eval()
        with torch.no_grad():
            dist, _ = model(to_tensor(obs, batch=True))
            return int(torch.argmax(dist.logits, dim=-1).item())

    eval_rewards, eval_mean, success_rate, selected_trajectory = evaluate_policy(a2c_policy, episodes=EVAL_EPISODES, seed_offset=EVAL_SEED_OFFSET)
    all_results["A2C"] = MethodResult(
        history=raw_rewards,
        eval_mean=eval_mean,
        success_rate=success_rate,
        best_reward=max(eval_rewards),
        selected_success=selected_trajectory.success,
        selected_steps=selected_trajectory.steps,
        description="Actor-critic method using an actor policy and critic value baseline.",
    )
    extra = (
        "The actor selects actions and the critic estimates value. "
        "A small supervised warm-start is used for a short Colab run; evaluation uses no forced action."
    )
    return (
        plot_training_curve(raw_rewards, "A2C: Raw Training Reward"),
        plot_trajectory(selected_trajectory, "A2C trajectory"),
        format_summary("A2C", raw_rewards, eval_rewards, success_rate, selected_trajectory, extra),
    )


# ──────────────────────────────────────────────────────────────────────────────
# Comparison and preset runner
# ──────────────────────────────────────────────────────────────────────────────


def generate_comparison_views():
    fig_result, ax_result = plt.subplots(figsize=(8.2, 4.8))
    has_data = False
    for method, result in all_results.items():
        if result is None or not result.history:
            continue
        has_data = True
        window = max(5, len(result.history) // 12)
        x = np.linspace(0.0, 100.0, len(result.history))
        ax_result.plot(x, moving_average(result.history, window=window), linewidth=2.3, label=method)
    if not has_data:
        ax_result.text(0.5, 0.5, "No completed run yet.\nTrain one or more methods first.", ha="center", va="center", transform=ax_result.transAxes, fontsize=12)
    else:
        ax_result.legend(loc="lower right")
    ax_result.set_title("Training Progress View — not a strict benchmark")
    ax_result.set_xlabel("Normalised training progress (%)")
    ax_result.set_ylabel("Raw episode reward, moving average")
    ax_result.grid(True, linestyle="--", alpha=0.42)
    fig_result.tight_layout()

    fig_eval, ax_eval = plt.subplots(figsize=(8.2, 4.8))
    completed = [(m, r) for m, r in all_results.items() if r is not None]
    if not completed:
        xs = np.linspace(float(OBS_LOW[0]), float(OBS_HIGH[0]), 400)
        ys = mountain_height(xs)
        ax_eval.plot(xs, ys, linewidth=2.4, label="Track")
        ax_eval.fill_between(xs, ys, ys.min() - 0.35, alpha=0.16)
        ax_eval.axvline(0.5, linestyle="--", linewidth=1.6, label="Finish line")
        ax_eval.scatter([-0.5], mountain_height(np.array([-0.5])), s=100, marker="o", label="Typical start valley")
        ax_eval.set_title("MountainCar Environment Structure")
        ax_eval.set_xlabel("Position")
        ax_eval.set_ylabel("Track height")
        ax_eval.set_xlim(float(OBS_LOW[0]), float(OBS_HIGH[0]))
        ax_eval.legend(loc="upper left")
    else:
        labels = [m for m, _ in completed]
        success_rates = [r.success_rate for _, r in completed]
        x = np.arange(len(labels))
        ax_eval.bar(x, success_rates)
        ax_eval.set_xticks(x)
        ax_eval.set_xticklabels(labels, rotation=20, ha="right")
        ax_eval.set_ylim(0, 105)
        ax_eval.set_ylabel("Success rate on common final evaluation seeds (%)")
        ax_eval.set_title("Fair Final Evaluation Scorecard")
        for idx, val in enumerate(success_rates):
            ax_eval.text(idx, val + 2, f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
    ax_eval.grid(True, axis="y", linestyle="--", alpha=0.42)
    fig_eval.tight_layout()

    support_notes = {
        "Behavioural Cloning": "Expert demonstrations + grid-labelled imitation data",
        "Q-Learning": "Momentum-prior Q initialisation + training reward shaping + guided exploration; final policy only",
        "REINFORCE": "Training reward shaping + supervised warm-start; final policy only",
        "A2C": "Training reward shaping + supervised warm-start; final policy only",
    }

    rows = []
    for method, result in all_results.items():
        support = support_notes.get(method, "—")
        if result is None:
            rows.append(f"| {method} | Not run | — | — | — | — | {support} |")
        else:
            traj_status = "Reached" if result.selected_success else "Not reached"
            rows.append(
                f"| {method} | Completed | {result.eval_mean:.2f} | {result.best_reward:.2f} | "
                f"{result.success_rate:.1f}% | {traj_status} in {result.selected_steps} steps | {support} |"
            )

    comparison_md = f"""
### Fairness-aware summary table

| Method | Status | Mean eval reward | Best eval reward | Success rate | Displayed trajectory | Training support / data |
|---|---:|---:|---:|---:|---:|---|
""" + "\n".join(rows) + f"""

**Fair evaluation rule:** checkpoint selection uses the same validation seed list starting at `{VALIDATION_SEED_OFFSET}`, and final metrics use the same `{EVAL_EPISODES}` evaluation episodes with the same seed list starting at `{EVAL_SEED_OFFSET}` for every method. Evaluation uses the original MountainCar reward only and no action forcing.

**Important fairness note:** the preset run is a fair **final-performance evaluation**, but it is not a pure training-budget benchmark because Behavioural Cloning uses expert data, Q-Learning uses guided exploration, and policy-gradient methods use a short warm-start. These supports are disclosed so the dashboard remains honest for presentation and learning.

**How to read reward:** MountainCar gives `-1` for every step. A car that reaches the finish line in 130 steps has reward about `-130`, which is better than `-200`.
"""
    return fig_result, fig_eval, comparison_md


def run_all_presets():
    started = time.time()
    run_bc(PRESETS["Behavioural Cloning"]["epochs"], PRESETS["Behavioural Cloning"]["lr"], PRESETS["Behavioural Cloning"]["expert_episodes"])
    run_q_learning(PRESETS["Q-Learning"]["episodes"], PRESETS["Q-Learning"]["lr"], PRESETS["Q-Learning"]["gamma"], PRESETS["Q-Learning"]["epsilon"], PRESETS["Q-Learning"]["bins"])
    run_reinforce(PRESETS["REINFORCE"]["episodes"], PRESETS["REINFORCE"]["lr"], PRESETS["REINFORCE"]["gamma"])
    run_a2c(PRESETS["A2C"]["episodes"], PRESETS["A2C"]["lr"], PRESETS["A2C"]["gamma"])
    fig_result, fig_env, comparison_md = generate_comparison_views()
    elapsed = time.time() - started
    comparison_md += f"\n\nPreset training completed in **{elapsed:.1f} seconds** on **{DEVICE}**."
    return fig_result, fig_env, comparison_md



# ──────────────────────────────────────────────────────────────────────────────
# Gradio user interface — Variant B visual layout
# ──────────────────────────────────────────────────────────────────────────────

CUSTOM_CSS = """
.gradio-container {max-width: 1340px !important; margin: auto;}
#variant-shell {
    padding: 20px 22px 10px 22px;
    border-radius: 26px;
    background: radial-gradient(circle at top left, #e0f2fe 0%, #f8fafc 34%, #eef2ff 100%);
    border: 1px solid #dbeafe;
    margin-bottom: 18px;
}
.variant-title {
    font-size: 38px;
    line-height: 1.08;
    font-weight: 850;
    margin: 0 0 8px 0;
    letter-spacing: -0.04em;
    color: #0f172a;
}
.variant-subtitle {
    font-size: 16px;
    color: #334155;
    max-width: 960px;
    margin: 0;
}
.variant-ribbon {
    display: inline-block;
    padding: 7px 12px;
    border-radius: 999px;
    background: #0f172a;
    color: #ffffff;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    margin-bottom: 12px;
}
.board-card {
    border-radius: 22px;
    border: 1px solid #e2e8f0;
    background: rgba(255,255,255,0.82);
    box-shadow: 0 12px 36px rgba(15,23,42,0.08);
    padding: 18px 18px;
    margin: 8px 0 14px 0;
}
.side-panel {
    border-radius: 20px;
    background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%);
    border: 1px solid #e5e7eb;
    padding: 16px;
    margin-bottom: 12px;
}
.side-panel h3 {margin-top: 0; color: #0f172a;}
.method-chip-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(0, 1fr));
    gap: 12px;
    margin: 14px 0 2px 0;
}
.method-chip {
    border: 1px solid #dbeafe;
    background: #ffffff;
    border-radius: 18px;
    padding: 14px;
    min-height: 98px;
}
.method-chip b {color: #0f172a; font-size: 15px;}
.method-chip span {display:block; color:#475569; font-size: 12.5px; margin-top: 6px;}
.rule-box {
    border-radius: 18px;
    background: #f8fafc;
    border-left: 6px solid #0f172a;
    padding: 14px 16px;
    color: #1e293b;
}
.workstation-note {
    border-radius: 16px;
    border: 1px solid #e2e8f0;
    padding: 12px 14px;
    margin-bottom: 10px;
    background: #ffffff;
}
.workstation-note b {color: #0f172a;}
button.primary {border-radius: 14px !important;}
@media (max-width: 900px) {
    .method-chip-grid {grid-template-columns: repeat(2, minmax(0, 1fr));}
    .variant-title {font-size: 30px;}
}
@media (max-width: 560px) {
    .method-chip-grid {grid-template-columns: 1fr;}
}
"""


def workstation_note(title: str, purpose: str, method_logic: str, output: str) -> str:
    return f"""
<div class='workstation-note'>
<b>{title}</b><br>
<b>Goal:</b> {purpose}<br>
<b>Learning logic:</b> {method_logic}<br>
<b>Displayed result:</b> {output}
</div>
"""


def landing_header() -> str:
    return """
<div id='variant-shell'>
    <div class='variant-ribbon'>Variant B · Policy Performance Studio</div>
    <h1 class='variant-title'>MountainCar Learning Arena</h1>
    <p class='variant-subtitle'>A redesigned control panel for training, testing, and presenting four learning methods on MountainCar-v0. The interface is intentionally arranged as a studio dashboard instead of the previous tab-first control screen.</p>
    <div class='method-chip-grid'>
        <div class='method-chip'><b>Behavioural Cloning</b><span>Imitation-based policy trained from expert state-action data.</span></div>
        <div class='method-chip'><b>Q-Learning</b><span>Discrete value-table learning with Bellman updates.</span></div>
        <div class='method-chip'><b>REINFORCE</b><span>Monte-Carlo policy-gradient learning from full returns.</span></div>
        <div class='method-chip'><b>A2C</b><span>Actor-critic learning using advantage estimates.</span></div>
    </div>
</div>
"""


def fairness_card() -> str:
    return """
<div class='rule-box'>
<b>Evaluation rule:</b> Every method is finally tested on the same MountainCar environment, same reward rule, same fixed evaluation seeds, and the same number of evaluation episodes. The car path shown is produced by the trained policy rollout, not by manual action forcing.
<br><br>
<b>Training note:</b> Method-specific training support is disclosed in the report because short Colab demonstrations need practical support for this sparse-reward environment.
</div>
"""


BLOCKS_KWARGS: Dict[str, Any] = {"title": "MountainCar Learning Arena"}
LAUNCH_STYLE_KWARGS: Dict[str, Any] = {}
try:
    theme_cls = getattr(gr.themes, "Monochrome", None) or getattr(gr.themes, "Soft")
    chosen_theme = theme_cls()
    block_params = inspect.signature(gr.Blocks).parameters
    if "theme" in block_params:
        BLOCKS_KWARGS.update({"theme": chosen_theme, "css": CUSTOM_CSS})
    else:
        LAUNCH_STYLE_KWARGS.update({"theme": chosen_theme, "css": CUSTOM_CSS})
except Exception:
    try:
        LAUNCH_STYLE_KWARGS.update({"theme": gr.themes.Soft(), "css": CUSTOM_CSS})
    except Exception:
        pass

with gr.Blocks(**BLOCKS_KWARGS) as demo:
    gr.HTML(landing_header())

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=310):
            gr.HTML("""
<div class='side-panel'>
<h3>Experiment Console</h3>
<p>Use the preset experiment for a complete submission-ready run, or open the method workstations below for individual training.</p>
<p><b>Default evaluation:</b> 50 common episodes.</p>
<p><b>Trajectory rule:</b> successful rollout is preferred when available.</p>
</div>
""")
            gr.HTML(fairness_card())
            run_all_btn = gr.Button("Run Complete Arena Experiment", variant="primary")
            all_summary = gr.Markdown(label="Arena report")
        with gr.Column(scale=2, min_width=520):
            gr.HTML("<div class='board-card'><b>Main dashboard output</b><br>After running the preset experiment, the two panels below show training progress and final evaluation success.</div>")
            with gr.Row():
                all_compare_plot = gr.Plot(label="Learning progress panel")
                all_env_plot = gr.Plot(label="Final success panel")
    run_all_btn.click(run_all_presets, inputs=[], outputs=[all_compare_plot, all_env_plot, all_summary])

    with gr.Accordion("Method Workstations", open=False):
        with gr.Tabs():
            with gr.TabItem("01 · Imitation Policy"):
                gr.HTML(workstation_note("Behavioural Cloning Workstation", "Train a policy by copying expert decisions.", "A neural classifier receives position and velocity and predicts the action class.", "Loss curve, final reward statistics, success rate, and rollout path."))
                with gr.Row():
                    with gr.Column(scale=1):
                        bc_epochs = gr.Slider(5, 120, value=PRESETS["Behavioural Cloning"]["epochs"], step=1, label="Epoch count")
                        bc_lr = gr.Slider(1e-4, 1e-2, value=PRESETS["Behavioural Cloning"]["lr"], step=1e-4, label="Step size")
                        bc_expert_eps = gr.Slider(10, 150, value=PRESETS["Behavioural Cloning"]["expert_episodes"], step=5, label="Expert rollout count")
                        bc_btn = gr.Button("Build Imitation Policy", variant="primary")
                    with gr.Column(scale=2):
                        bc_plot = gr.Plot(label="Imitation training panel")
                        bc_rollout = gr.Plot(label="Imitation rollout path")
                        bc_result = gr.Markdown()
                bc_btn.click(run_bc, inputs=[bc_epochs, bc_lr, bc_expert_eps], outputs=[bc_plot, bc_rollout, bc_result])

            with gr.TabItem("02 · Value Table"):
                gr.HTML(workstation_note("Q-Learning Workstation", "Learn a value table for discretised position and velocity states.", "Bellman target updates improve the Q-value of each state-action pair.", "Reward progression, greedy evaluation path, and score summary."))
                with gr.Row():
                    with gr.Column(scale=1):
                        q_episodes = gr.Slider(500, 6000, value=PRESETS["Q-Learning"]["episodes"], step=100, label="Training rollouts")
                        q_lr = gr.Slider(0.01, 0.30, value=PRESETS["Q-Learning"]["lr"], step=0.01, label="Alpha")
                        q_gamma = gr.Slider(0.90, 0.999, value=PRESETS["Q-Learning"]["gamma"], step=0.001, label="Gamma")
                        q_epsilon = gr.Slider(0.10, 1.00, value=PRESETS["Q-Learning"]["epsilon"], step=0.05, label="Starting exploration")
                        q_bins = gr.Slider(15, 70, value=PRESETS["Q-Learning"]["bins"], step=5, label="State grid density")
                        q_btn = gr.Button("Train Value Table", variant="primary")
                    with gr.Column(scale=2):
                        q_plot = gr.Plot(label="Value-table learning panel")
                        q_rollout = gr.Plot(label="Greedy rollout path")
                        q_result = gr.Markdown()
                q_btn.click(run_q_learning, inputs=[q_episodes, q_lr, q_gamma, q_epsilon, q_bins], outputs=[q_plot, q_rollout, q_result])

            with gr.TabItem("03 · Policy Gradient"):
                gr.HTML(workstation_note("REINFORCE Workstation", "Optimise a stochastic policy using complete episode returns.", "Sampled action log-probabilities are weighted by discounted reward-to-go.", "Training reward trend, final greedy path, and evaluation report."))
                with gr.Row():
                    with gr.Column(scale=1):
                        rf_episodes = gr.Slider(100, 2500, value=PRESETS["REINFORCE"]["episodes"], step=50, label="Policy update episodes")
                        rf_lr = gr.Slider(1e-4, 5e-3, value=PRESETS["REINFORCE"]["lr"], step=1e-4, label="Policy learning rate")
                        rf_gamma = gr.Slider(0.90, 0.999, value=PRESETS["REINFORCE"]["gamma"], step=0.001, label="Return discount")
                        rf_btn = gr.Button("Train Policy Gradient", variant="primary")
                    with gr.Column(scale=2):
                        rf_plot = gr.Plot(label="Policy-gradient learning panel")
                        rf_rollout = gr.Plot(label="Policy rollout path")
                        rf_result = gr.Markdown()
                rf_btn.click(run_reinforce, inputs=[rf_episodes, rf_lr, rf_gamma], outputs=[rf_plot, rf_rollout, rf_result])

            with gr.TabItem("04 · Actor Critic"):
                gr.HTML(workstation_note("A2C Workstation", "Train an actor policy with help from a critic value baseline.", "The critic estimates state value and the actor updates using advantage information.", "Learning curve, learned trajectory, and evaluation metrics."))
                with gr.Row():
                    with gr.Column(scale=1):
                        a2c_episodes = gr.Slider(100, 2500, value=PRESETS["A2C"]["episodes"], step=50, label="Actor-critic episodes")
                        a2c_lr = gr.Slider(1e-4, 5e-3, value=PRESETS["A2C"]["lr"], step=1e-4, label="Shared learning rate")
                        a2c_gamma = gr.Slider(0.90, 0.999, value=PRESETS["A2C"]["gamma"], step=0.001, label="Future reward weight")
                        a2c_btn = gr.Button("Train Actor-Critic", variant="primary")
                    with gr.Column(scale=2):
                        a2c_plot = gr.Plot(label="Actor-critic learning panel")
                        a2c_rollout = gr.Plot(label="Actor rollout path")
                        a2c_result = gr.Markdown()
                a2c_btn.click(run_a2c, inputs=[a2c_episodes, a2c_lr, a2c_gamma], outputs=[a2c_plot, a2c_rollout, a2c_result])

    with gr.Accordion("Comparison Report Builder", open=True):
        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                gr.HTML("""
<div class='side-panel'>
<h3>Report Builder</h3>
<p>Generate the comparison after running one or more methods. This section gives a table plus the same fairness notes needed for explanation.</p>
</div>
""")
                compare_btn = gr.Button("Refresh Final Report", variant="secondary")
                compare_table = gr.Markdown()
            with gr.Column(scale=2):
                with gr.Row():
                    results_chart = gr.Plot(label="Normalised learning comparison")
                    env_chart = gr.Plot(label="Common-seed evaluation")
        compare_btn.click(generate_comparison_views, inputs=[], outputs=[results_chart, env_chart, compare_table])


def launch_app() -> None:
    try:
        import google.colab  # type: ignore  # noqa: F401
        in_colab = True
    except Exception:
        in_colab = False

    app = demo.queue()
    if in_colab:
        app.launch(share=True, debug=True, show_error=True, **LAUNCH_STYLE_KWARGS)
    else:
        app.launch(server_name="127.0.0.1", server_port=7860, inbrowser=True, show_error=True, **LAUNCH_STYLE_KWARGS)


if __name__ == "__main__":
    launch_app()
